In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "/workspaces/dev/modules/python-utils",
    "/workspaces/dev/modules/ai-utils",
    "/workspaces/dev/test/performance_test/libri",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path
import optuna
from optuna.trial import Trial
import joblib
import random
import time
import optuna.visualization as vis

In [ ]:
from sj_ai_utils.datasets.libri_speech_asr_corpus import trans_txt_to_sclite_trn, search_all_data
from sj_ai_utils.evaluator.sclite_utils import TRNFormat, sclite_trn, parse_sclite_summary
from sj_utils.collection import SafetyDict
from sj_utils.evaluator import TimeChecker
from sj_utils.file.yaml import YamlSaver

In [ ]:
from util import get_token_saver_loader_transcriber, normalize_text

In [ ]:
DESCRIPTION = """
20250722 기점으로 수정된 RT Whisper 알고리즘을 사용함
saver_loader을 사용하여 측정되었으며, prompt는 사용하지 않음
overlap_duration: 16000~112000, 16000 단위로 설정
배치 16
20250722/001 결과 기반 하이퍼파라미터 탐색 범위를 좁힘
"""

In [ ]:
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/train/"
STORAGE = "/workspaces/dev/storage/libri/"
STUDY = "/workspaces/dev/study/libri"
OUTPUT = "/workspaces/dev/hyperparameters/libri/20250723/001"

In [ ]:
src = Path(SOURCE)
storage = Path(STORAGE)
study_path = Path(STUDY) / "20250723_test.pkl"
study_path.parent.mkdir(parents=True, exist_ok=True)
output = Path(OUTPUT)
output.mkdir(parents=True, exist_ok=True)

In [ ]:
yaml_saver = YamlSaver(DESCRIPTION)

In [ ]:
src_folder = search_all_data(src)
len(src_folder)

In [ ]:
transcriber = get_token_saver_loader_transcriber(src, storage, SAMPLE_RATE)

In [ ]:
def objective(trial:Trial):
    hyperparameters = SafetyDict({
        "whisper": {
            "model_options": {
                "model_size_or_path": "large-v3",
                "device": "cuda",
                "compute_type": "float16",
            },
            "transcribe_options": {
                "beam_size":5,
                "vad_filter": False,
                "temperature": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
            }
        },
        "silero_vad": {
            "model_options": {},
            "run_options": {}
        },
        "asr": {
            "max_overlap_duration": trial.suggest_int(
                "overlap_duration", 16000, 112000, step=16000
            )
        },
        "position_weighted_filter": {
            # "boundary":trial.suggest_int("boundary", 0, 32000, step=100)
            "boundary":trial.suggest_int("boundary", 0, 64000, step=100)
        },
        "duration_filter":{
            "z_thresh":{
                "default": 2.0,
                # "en": trial.suggest_float("df_z_thresh", 0, 10.0, step = 0.1)
            },
            "min_dur": {
                "default": 160,
                # "en": 160
                # "en": trial.suggest_float("df_min_duration", 0, 8000, step = 160)
                "en": trial.suggest_float("df_min_duration", 0, 2080, step = 160)
            },
        },
        "probability_filter":{
            # "z_thresh":{
            #     "default": 3.0,
            #     "en": trial.suggest_float("pf_z_thresh", 0.0, 10.0, step = 0.1)
            # },
            "min_prob": {
                "default": 1.0,
                "en": trial.suggest_float("pf_min_prob", 0, 0.6, step = 0.01)
            },
        },
        "selector":{
            "iou_threshold": {
                "default": 0.5,
                # "en": trial.suggest_float("iou_threshold", 0, 1, step=0.01)
                "en": trial.suggest_float("iou_threshold", 0.05, 0.5, step=0.01)
            },
            "cos_threshold":{
                "default": 0.5,
                # "en": trial.suggest_float("cos_threshold", 0, 1, step=0.01)
                "en": trial.suggest_float("cos_threshold", 0.5, 0.85, step=0.01)
            },
            "padding": {
                "default": 3200,
                # "en": 3200
                # "en": trial.suggest_int("padding", 0, 32000, step=100)
                "en": trial.suggest_int("padding", 0, 3000, step=100)
            },
        },
    })

    samples = random.sample(src_folder, 16)

    data = {}
    for sample in samples:
        trans_txt = next(sample.glob("*.trans.txt"))
        ref = trans_txt_to_sclite_trn(trans_txt, normalize_text)
        hyp = [transcriber(
            flac,
            TimeChecker(),
            hyperparameters,
            hyperparameters["asr"]["max_overlap_duration"],
        ) for flac in sorted(sample.glob("*.flac"))]
        data[sample.stem] = {"ref": ref,"hyp": hyp}

    concat_result = {}
    for value in data.values():
        for k, v in value.items():
            if k not in concat_result:
                concat_result[k] = []
            concat_result[k].extend(v)

    output = sclite_trn(concat_result["ref"], concat_result["hyp"])
    result = parse_sclite_summary(output)

    return result["wer_percent"]

In [ ]:
if study_path.exists():
    study = joblib.load(study_path)
else:
    study = optuna.create_study(direction="minimize")

In [ ]:
for _ in range(1):
    study.optimize(objective, n_trials=5)
    joblib.dump(study, study_path)

In [ ]:
def show_and_save_plot(study:optuna.Study, extra_name:str = ""):
    if extra_name:
        extra_name = f"_{extra_name}"

    fig_pi = vis.plot_param_importances(study)
    fig_pi.write_html(output / f"param_importances{extra_name}_{time.strftime('%Y%m%d_%H%M%S')}.html")
    fig_pi.show()

    fig_ps = vis.plot_slice(study)
    fig_ps.write_html(output / f"param_slices{extra_name}_{time.strftime('%Y%m%d_%H%M%S')}.html")
    fig_ps.show()

    fig_ppc = vis.plot_parallel_coordinate(study)
    fig_ppc.write_html(output / f"parallel_coordinate{extra_name}_{time.strftime('%Y%m%d_%H%M%S')}.html")
    fig_ppc.show()

In [ ]:
max10study = optuna.create_study(direction=study.direction)
max10study.add_trials([t for t in study.trials if t.values[0] <= 10])
max15study = optuna.create_study(direction=study.direction)
max15study.add_trials([t for t in study.trials if t.values[0] <= 15])
max20study = optuna.create_study(direction=study.direction)
max20study.add_trials([t for t in study.trials if t.values[0] <= 20])

show_and_save_plot(study)
show_and_save_plot(max10study, "max10")
show_and_save_plot(max15study, "max15")
show_and_save_plot(max20study, "max20")

In [ ]:
completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

In [ ]:
top_trials = sorted(completed_trials, key=lambda t: t.values[0])[:30]
for i, trial in enumerate(top_trials):
    data = {
        "whisper": {
            "model_options": {
                "model_size_or_path": "large-v3",
                "device": "cuda",
                "compute_type": "float16",
            },
            "transcribe_options": {
                "beam_size":5,
                "vad_filter": False,
                "temperature": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
            }
        },
        "silero_vad": {
            "model_options": {},
            "run_options": {}
        },
        "asr": {
            "max_overlap_duration":trial.params["overlap_duration"]
        },
        "position_weighted_filter": {
            "boundary":trial.params["boundary"]
        },
        "duration_filter":{
            "z_thresh":{
                "default": 2.0,
                # "en": trial.params["df_z_thresh"]
            },
            "min_dur": {
                "default": 160,
                "en": trial.params["df_min_duration"]
            },
        },
        "probability_filter":{
            "z_thresh":{
                "default": 3.0,
                # "en": trial.params["pf_z_thresh"]
            },
            "min_prob": {
                "default": 1.0,
                "en": trial.params["pf_min_prob"]
            },
        },
        "selector":{
            "iou_threshold": {
                "default": 0.5,
                "en": trial.params["iou_threshold"]
            },
            "cos_threshold":{
                "default": 0.5,
                "en": trial.params["cos_threshold"]
            },
            "padding": {
                "default": 3200,
                "en": trial.params["padding"]
            },
        },
    }
    yaml_saver.save(
        data,
        output / f"trial_wer{str(trial.values[0]).replace('.', 'o')}_{trial.number}_{time.strftime('%Y%m%d_%H%M%S')}.yaml"
    )